# Home Run Bets Tracker
Track daily HR bets on ProphetX and Novig. Log results, monitor P&L, and refine predictions.

**Bet sizes:**
- Singles: $10
- Parlays: $5

**Platforms:** ProphetX | Novig

---

## 1. Setup

In [ ]:
import sqlite3
import pandas as pd
from datetime import date
import os
import json

DB_PATH = 'data/bets.db'

def get_conn():
    return sqlite3.connect(DB_PATH)

def init_db():
    with get_conn() as conn:
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS singles (
                id               INTEGER PRIMARY KEY AUTOINCREMENT,
                bet_date         TEXT NOT NULL,
                platform         TEXT NOT NULL,    -- 'prophetx' or 'novig'
                player           TEXT NOT NULL,
                game             TEXT,             -- e.g. 'ARI @ BAL 5:35 PM'
                wager            REAL NOT NULL DEFAULT 10.0,
                odds             TEXT,             -- e.g. '+360'
                potential_payout REAL,             -- 'To Win' amount shown on platform
                result           TEXT,             -- 'win', 'loss', or NULL if pending
                payout           REAL,             -- total return received if won (wager + profit)
                notes            TEXT
            );
        """)
    print('DB ready:', DB_PATH)

init_db()

## 2. Log Today's Bets

In [ ]:
def log_singles(bet_date, platform, bets, wager=10.0):
    """
    Log multiple single HR bets at once.
    bets: list of dicts with keys: player, game, odds, potential_payout
    """
    rows = []
    for b in bets:
        rows.append((
            bet_date,
            platform.lower(),
            b['player'],
            b.get('game'),
            wager,
            b.get('odds'),
            b.get('potential_payout'),
        ))
    with get_conn() as conn:
        conn.executemany(
            '''INSERT INTO singles
               (bet_date, platform, player, game, wager, odds, potential_payout)
               VALUES (?,?,?,?,?,?,?)''',
            rows
        )
    print(f'Logged {len(rows)} singles on {platform} for {bet_date}')


# ── April 14, 2026 ──────────────────────────────────────────────────────────
TODAY = '2026-04-14'

log_singles(TODAY, 'prophetx', [
    {'player': 'Gunnar Henderson', 'game': 'ARI @ BAL 5:35 PM', 'odds': '+360', 'potential_payout': 36.00},
    {'player': 'Willy Adames',     'game': 'SF @ CIN 5:40 PM',  'odds': '+380', 'potential_payout': 38.00},
    {'player': 'Bryce Harper',     'game': 'CHC @ PHI 5:40 PM', 'odds': '+340', 'potential_payout': 34.00},
    {'player': 'Yordan Alvarez',   'game': 'COL @ HOU 7:10 PM', 'odds': '+260', 'potential_payout': 26.00},
    {'player': 'Kyle Schwarber',   'game': 'CHC @ PHI 5:40 PM', 'odds': '+200', 'potential_payout': 20.00},
    {'player': 'Pete Alonso',      'game': 'ARI @ BAL 5:35 PM', 'odds': '+355', 'potential_payout': 35.50},
    {'player': 'Oneil Cruz',       'game': 'WSH @ PIT 5:40 PM', 'odds': '+340', 'potential_payout': 34.00},
    {'player': 'Hunter Goodman',   'game': 'COL @ HOU 7:10 PM', 'odds': '+405', 'potential_payout': 40.50},
])

## 3. Record Results

In [ ]:
def set_result(bet_date, player, result, payout=None):
    """
    Update result for a single bet.
    result  = 'win' or 'loss'
    payout  = total return received (wager + profit), e.g. $46 on a $10 bet at +360
              Leave None for a loss.
    """
    with get_conn() as conn:
        conn.execute(
            'UPDATE singles SET result=?, payout=? WHERE bet_date=? AND player=?',
            (result, payout, bet_date, player)
        )
    tag = f'  total return=${payout:.2f}' if payout else ''
    print(f'{player} ({bet_date}): {result}{tag}')


# ── Fill in once games are final ─────────────────────────────────────────────
# Wins: pass total return (wager + profit).  Losses: just pass 'loss'.
#
# set_result(TODAY, 'Gunnar Henderson', 'win', payout=46.00)   # $10 + $36
# set_result(TODAY, 'Willy Adames',     'loss')
# set_result(TODAY, 'Bryce Harper',     'loss')
# set_result(TODAY, 'Yordan Alvarez',   'loss')
# set_result(TODAY, 'Kyle Schwarber',   'loss')
# set_result(TODAY, 'Pete Alonso',      'loss')
# set_result(TODAY, 'Oneil Cruz',       'loss')
# set_result(TODAY, 'Hunter Goodman',   'loss')

print('Result function ready — uncomment and fill in after games complete.')

## 4. View Bets

In [ ]:
def show_bets(bet_date=None):
    """Display bets. Pass a date string to filter, or None for all."""
    with get_conn() as conn:
        q = 'SELECT * FROM singles'
        params = []
        if bet_date:
            q += ' WHERE bet_date=?'
            params = [bet_date]
        df = pd.read_sql(q, conn, params=params)
    if not df.empty:
        display(df)
    else:
        print('No bets found.')

show_bets(TODAY)

## 5. P&L Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def pnl_summary():
    with get_conn() as conn:
        df = pd.read_sql('SELECT * FROM singles WHERE result IS NOT NULL', conn)

    if df.empty:
        print('No settled bets yet.')
        return

    df = df.copy()
    df['payout'] = df['payout'].fillna(0)
    df['pnl'] = df.apply(
        lambda r: (r['payout'] - r['wager']) if r['result'] == 'win' else -r['wager'],
        axis=1
    )

    wins   = (df['result'] == 'win').sum()
    total  = len(df)
    wagered = df['wager'].sum()
    net_pnl = df['pnl'].sum()
    win_pct = wins / total * 100 if total else 0

    print('=== P&L SUMMARY ===')
    print(f'Record:  {wins}W - {total - wins}L  ({win_pct:.1f}% win rate)')
    print(f'Wagered: ${wagered:.2f}')
    print(f'P&L:     ${net_pnl:+.2f}')
    print(f'ROI:     {net_pnl / wagered * 100:+.1f}%' if wagered else '')

    # Cumulative P&L chart
    daily = df.groupby('bet_date')['pnl'].sum().reset_index().sort_values('bet_date')
    daily['cumulative'] = daily['pnl'].cumsum()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(daily['bet_date'], daily['pnl'],
           color=['green' if v >= 0 else 'red' for v in daily['pnl']],
           alpha=0.6, label='Daily P&L')
    ax.plot(daily['bet_date'], daily['cumulative'],
            marker='o', color='steelblue', linewidth=2, label='Cumulative P&L')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title('P&L Over Time')
    ax.set_xlabel('Date')
    ax.set_ylabel('P&L ($)')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

pnl_summary()

## 5b. Load Historical Bets

In [ ]:
def load_historical(bets):
    """
    Bulk-insert historical bets with known results.
    Skips any bet already in the DB for that date+player combo.
    """
    inserted = 0
    skipped  = 0
    with get_conn() as conn:
        for b in bets:
            exists = conn.execute(
                'SELECT 1 FROM singles WHERE bet_date=? AND player=?',
                (b['bet_date'], b['player'])
            ).fetchone()
            if exists:
                skipped += 1
                continue
            conn.execute(
                '''INSERT INTO singles
                   (bet_date, platform, player, game, wager, odds, potential_payout, result, payout, notes)
                   VALUES (?,?,?,?,?,?,?,?,?,?)''',
                (
                    b['bet_date'], b.get('platform', 'prophetx'), b['player'],
                    b.get('game'), b.get('wager', 10.0), b.get('odds'),
                    b.get('potential_payout'), b.get('result'), b.get('payout'),
                    b.get('notes'),
                )
            )
            inserted += 1
    print(f'Loaded {inserted} historical bets ({skipped} already existed, skipped).')


HISTORICAL = [
    # ── Sunday Apr 12 ──────────────────────────────────────────────────────
    {'bet_date': '2026-04-12', 'player': 'Shohei Ohtani',         'game': 'TEX @ LAD 5:05 PM',
     'odds': '+240', 'potential_payout': 24.00, 'result': 'win',  'payout': 34.00},

    {'bet_date': '2026-04-12', 'player': 'Vladimir Guerrero Jr.', 'game': 'MIA @ TOR 12:37 PM',
     'odds': '+540', 'potential_payout': 54.00, 'result': 'loss'},

    {'bet_date': '2026-04-12', 'player': 'Kyle Schwarber',        'game': 'ARI @ PHI 12:35 PM',
     'odds': '+280', 'potential_payout': 28.00, 'result': 'loss'},

    {'bet_date': '2026-04-12', 'player': 'Aaron Judge',           'game': 'NYY @ TB 12:45 PM',
     'odds': '+305', 'potential_payout': 30.50, 'result': 'win',  'payout': 40.50},

    {'bet_date': '2026-04-12', 'player': 'Jordan Walker',         'game': 'BOS @ STL 1:15 PM',
     'odds': '+770', 'potential_payout': 77.00, 'result': 'win',  'payout': 87.00},

    {'bet_date': '2026-04-12', 'player': 'Oneil Cruz',            'game': 'PIT @ CHC 1:20 PM',
     'odds': '+250', 'potential_payout': 25.00, 'result': 'win',  'payout': 35.00},

    {'bet_date': '2026-04-12', 'player': 'Ian Happ',              'game': 'PIT @ CHC 1:20 PM',
     'odds': '+330', 'potential_payout': 33.00, 'result': 'loss'},

    # ── Monday Apr 13 ──────────────────────────────────────────────────────
    {'bet_date': '2026-04-13', 'player': 'Shohei Ohtani',         'game': 'NYM @ LAD 5:10 PM',
     'odds': '+345', 'potential_payout': 34.50, 'result': 'loss'},

    {'bet_date': '2026-04-13', 'player': 'Tyger Ward',            'game': 'ARI @ BAL 5:25 PM',
     'odds': '+440', 'potential_payout': 44.00, 'result': 'loss'},

    {'bet_date': '2026-04-13', 'player': 'Matt Olson',            'game': 'MIA @ ATL 6:15 PM',
     'odds': '+395', 'potential_payout': 39.50, 'result': 'loss'},

    {'bet_date': '2026-04-13', 'player': 'Aaron Judge',           'game': 'LAA @ NYY 5:05 PM',
     'odds': '+220', 'potential_payout': 22.00, 'result': 'win',  'payout': 32.00},

    {'bet_date': '2026-04-13', 'player': 'Kyle Schwarber',        'game': 'CHC @ PHI 5:40 PM',
     'odds': '+235', 'potential_payout': 23.50, 'result': 'win',  'payout': 33.50},

    {'bet_date': '2026-04-13', 'player': 'Gunnar Henderson',      'game': 'ARI @ BAL 5:25 PM',
     'odds': '+445', 'potential_payout': 44.50, 'result': 'loss'},

    {'bet_date': '2026-04-13', 'player': 'Zach Neto',             'game': 'LAA @ NYY 5:05 PM',
     'odds': '+500', 'potential_payout': 50.00, 'result': 'loss'},
]

load_historical(HISTORICAL)

### 6a. Bet Tracker Agent — P&L and results

### 6b. Predictor Agent — BallparkPal picks

In [ ]:
# Example: generate today's HR picks
homer = Homer()
response = homer.run(
    "Fetch today's BallparkPal projections and give me the top 8 HR picks "
    "with confidence levels and reasons. Cross-reference against our historical win rates."
)
print(response)

### 6c. Overseer Agent — full daily workflow

### 6d. Daily Workflow — Run Everything

### 6e. Auto-Log from Picks

Generates a ready-to-fill `log_singles()` call from today's AI picks.

1. Run the cell below to get structured picks
2. Fill in the actual odds + game from ProphetX or Novig
3. Uncomment the platform you're betting on and run

In [ ]:
from datetime import date

def bet_slip(picks: list, platform: str = "prophetx", wager: float = 10.0) -> None:
    """
    Print a ready-to-fill log_singles() call from a structured picks list.
    Fill in the odds and game fields from your platform before running.

    Args:
        picks    : list of dicts from homer.get_picks_json()
        platform : 'prophetx' or 'novig'
        wager    : bet size per single (default $10)
    """
    today = date.today().isoformat()
    print(f"# ── {platform.upper()} — {today} ─────────────────────────────")
    print(f"log_singles('{today}', '{platform}', [")
    for p in picks:
        player     = p.get("player", "")
        matchup    = p.get("matchup", "")
        confidence = p.get("confidence", "")
        reasoning  = p.get("reasoning", "")
        print(f"    # {confidence} — {reasoning}")
        print(f"    {{'player': '{player}', 'game': '{matchup}', "
              f"'odds': '___', 'potential_payout': 0.00}},")
    print(f"], wager={wager})")


homer = Homer()

print("Fetching today's picks (takes ~30s)...\n")
picks = homer.get_picks_json(top_n=8)

if not picks:
    print("No structured picks returned — run homer.run() for the full narrative output.")
else:
    print(f"Got {len(picks)} picks:\n")
    for i, p in enumerate(picks, 1):
        print(f"  {i}. [{p['confidence']}] {p['player']} ({p['matchup']})")
        print(f"     {p['reasoning']}")
    print()
    print("─" * 60)
    print("PROPHETX BET SLIP (fill in odds + game, then run log_singles):")
    print("─" * 60)
    bet_slip(picks, platform="prophetx")
    print()
    print("─" * 60)
    print("NOVIG BET SLIP (fill in odds + game, then run log_singles):")
    print("─" * 60)
    bet_slip(picks, platform="novig")

## 7. Backtesting

Scores every settled bet using reconstructed predictive criteria and correlates each factor against actual win/loss outcomes.

**Scoring factors (0–3 each, max 15 total):**
| Factor | Source | 0 pts | 1 pt | 2 pts | 3 pts |
|---|---|---|---|---|---|
| Barrel rate | Baseball Savant season | <5% | 5–10% | 10–15% | >15% |
| Hard hit % | Baseball Savant season | <40% | 40–45% | 45–50% | >50% |
| HR/FB ratio | Baseball Savant season | <10% | 10–15% | 15–20% | >20% |
| Recent form | HRs in last 14 days | 0 | 1 | 2 | 3+ |
| Odds | Market implied prob | +450+ | +350–449 | +250–349 | <+250 |

Higher composite score = stronger pre-bet signal. The report shows which factors actually predicted wins.

In [ ]:
from agents import run_backtest, backtest_report

# Score all settled bets and generate the full report + charts
# Note: makes Baseball Savant API calls for each player — takes ~30 seconds
df_backtest = backtest_report()